# Stage 2: Cohort Selection & Downsampling

## Goal

The full 13M row dataset spans 17 months and is too large for rapid development. We will:

1. **Isolate the final 3 consecutive months**: March, April, and May 2016
   - These are the most recent months in the training data
   - They contain the richest signal and most recent customer behavior

2. **Filter to active recurring customers**: Keep only customers who appear in ALL 3 months
   - Ensures we can compute lag features reliably
   - Customers missing from any month cannot have month-over-month comparisons

3. **Expected output**: ~600,000 active customers × 3 months = ~1.8M rows
   - Manageable size for rapid iteration and model development
   - Maintains the temporal structure needed for supervised learning

## Step 1: Import Libraries and Configure Logging

We import:
- **pandas**: DataFrames and data manipulation
- **pathlib.Path**: Cross-platform absolute path handling
- **logging**: Track processing progress
- **warnings**: Suppress non-critical alerts

We load from the **Parquet file saved in Stage 1**, not the raw CSV:
- Much faster (compressed, binary format vs text)
- Already has correct dtypes from Stage 1
- No need to re-parse 13M rows

In [ ]:
import pandas as pd
from pathlib import Path
import logging
import warnings

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings('ignore')

# Configure logging with timestamp and level for tracking progress
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

print('✓ Libraries loaded successfully')

## Step 2: Define Paths and Target Dates

**Simple Path Resolution**:
- The notebook is always in the `notebooks/` subdirectory
- We go one level up using `Path.cwd().parent` to reach project root
- Build absolute paths from there
- Assert that the input file exists before proceeding

**Target Dates**:
- **2016-03-28** = Month t-2
- **2016-04-28** = Month t-1
- **2016-05-28** = Month t (latest available)

In [ ]:
import os
from pathlib import Path

# The notebook is in notebooks/ so we go one level up to reach project root
# This is simpler and more reliable than searching parent directories
PROJECT_ROOT = Path.cwd().parent

# Build paths relative to project root
PARQUET_PATH = PROJECT_ROOT / 'data' / 'parquet' / 'train.parquet'
COHORT_OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'cohort.parquet'

# Create output directory if it does not exist
COHORT_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Define the 3 target monthly snapshots as exact pandas Timestamps
TARGET_DATES = [
    pd.Timestamp('2016-03-28'),  # Month t-2
    pd.Timestamp('2016-04-28'),  # Month t-1
    pd.Timestamp('2016-05-28')   # Month t (latest)
]

# Verify the parquet file actually exists before proceeding
assert PARQUET_PATH.exists(), f"Parquet file not found at: {PARQUET_PATH}"

print(f'Project root : {PROJECT_ROOT}')
print(f'Parquet path : {PARQUET_PATH}')
print(f'Parquet exists: {PARQUET_PATH.exists()}')
print(f'Output path  : {COHORT_OUTPUT_PATH}')
print(f'Target dates : {TARGET_DATES}')

## Step 3: Load Parquet File

We load the full parquet file saved in Stage 1:
- Much faster than re-reading the CSV (binary compressed format)
- Preserves dtypes from Stage 1 ingestion
- Expected shape: (13M+, 48 columns)
- Loading should take under 30 seconds

In [ ]:
# Load the parquet file using absolute path (always works)
df = pd.read_parquet(str(PARQUET_PATH), engine='pyarrow')

# Log the shape and date range
logging.info(f'Loaded parquet file. Shape: {df.shape}')
logging.info(f'Unique dates in dataset: {sorted(df["fecha_dato"].unique())}')

# Display sample rows
print('\nFirst 3 rows:')
df.head(3)

## Step 4: Filter to Target Date Window

We slice the dataframe to keep only rows matching our 3 target dates:
- Immediately reduces 13M rows down to roughly 3M rows
- Uses `.isin()` for efficient multi-value filtering
- Verifies all 3 dates are present before proceeding

In [ ]:
# Filter to only rows where fecha_dato matches one of the target dates
df_window = df[df['fecha_dato'].isin(TARGET_DATES)].copy()

# Log the filtered shape
logging.info(f'After filtering to target dates. Shape: {df_window.shape}')

# Verify all 3 dates are present
logging.info(f'Date distribution:\n{df_window["fecha_dato"].value_counts().sort_index()}')

# Assert that we have exactly 3 dates
assert df_window['fecha_dato'].nunique() == 3, "Expected 3 dates, got different count"
print('✓ All 3 target dates present')

## Step 5: Isolate Active Recurring Customers

We identify customers who appear in ALL 3 months:
- A customer missing from any month cannot have lag features computed later
- We group by `ncodpers` and keep only groups with exactly 3 records
- Expected result: roughly 600,000 unique active customers

In [ ]:
# Group by customer ID and count records per customer
customer_counts = df_window.groupby('ncodpers').size()

# Identify customers with exactly 3 records (appear in all 3 months)
active_customers = customer_counts[customer_counts == 3].index

# Filter df_window to only rows where ncodpers is in active_customers
df_cohort = df_window[df_window['ncodpers'].isin(active_customers)].copy()

# Log the results
logging.info(f'Active recurring customers: {df_cohort["ncodpers"].nunique():,}')
logging.info(f'Cohort shape (active customers only): {df_cohort.shape}')

# Verify all customers have exactly 3 records
records_per_customer = df_cohort.groupby('ncodpers').size().unique().tolist()
assert records_per_customer == [3], "All customers must have exactly 3 records"
print('✓ All customers in cohort have exactly 3 records')

## Step 6: Sort and Save Cohort

We sort by customer ID then date—critical for Stage 3:
- Stage 3 computes month-over-month differences and requires chronological order
- Enables efficient lag feature computation
- Save to parquet for fast loading in Stage 3

In [ ]:
# Sort by customer ID, then by date (chronological within each customer)
df_cohort = df_cohort.sort_values(['ncodpers', 'fecha_dato']).reset_index(drop=True)

# Save to parquet with compression using absolute path
df_cohort.to_parquet(str(COHORT_OUTPUT_PATH), engine='pyarrow', compression='snappy')

# Log final statistics
logging.info(f'Final cohort shape: {df_cohort.shape}')

# Calculate memory usage in MB
memory_mb = df_cohort.memory_usage(deep=True).sum() / 1024**2
logging.info(f'Cohort memory usage: {memory_mb:.2f} MB')

print(f'\n✓ Stage 2 Complete. Cohort saved to: {COHORT_OUTPUT_PATH}')

## Step 7: Cohort Validation Summary

Final sanity checks on the saved cohort:
- **Shape**: Confirm ~1.8M rows and 48 columns
- **Dates**: Verify exactly 3 dates present
- **Customers**: Confirm ~600,000 unique customers
- **Records per customer**: All should have exactly 3
- **Memory**: Confirm reasonable size for processing

All checks passing indicates readiness for Stage 3.

In [ ]:
# Reload df_cohort from COHORT_OUTPUT_PATH to confirm the parquet saved correctly
df_cohort = pd.read_parquet(str(COHORT_OUTPUT_PATH), engine='pyarrow')

print('\n=== COHORT VALIDATION SUMMARY ===')

# Check 1: Shape
print(f'\n1. Shape (expect ~1.8M rows, 48 cols):')
print(f'   {df_cohort.shape}')

# Check 2: Dates
print(f'\n2. Unique dates (expect exactly 3):')
print(f'   {sorted(df_cohort["fecha_dato"].unique())}')

# Check 3: Customer count
print(f'\n3. Unique customers (expect ~600,000):')
print(f'   {df_cohort["ncodpers"].nunique():,}')

# Check 4: Records per customer
print(f'\n4. Records per customer (expect all 3s):')
print(df_cohort.groupby('ncodpers').size().value_counts())

# Check 5: Memory usage in MB
memory_mb = df_cohort.memory_usage(deep=True).sum() / 1024**2
print(f'\n5. Memory usage: {memory_mb:.2f} MB')

print('\n✓ Cohort validation passed. Ready for Stage 3: Target Engineering')